# StockSage AI — XGBoost Training
Run on **Kaggle Notebooks** (free T4 GPU, 30 hrs/week).  
This notebook trains an XGBoost classifier to predict 7-day stock direction on Nifty 200 universe.

In [ ]:
# Install dependencies (Kaggle already has most)
!pip install yfinance pandas-ta xgboost scikit-learn joblib --quiet
# TA-Lib (optional)
try:
    import talib
except ImportError:
    print('TA-Lib not available; skipping candlestick features')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

NIFTY_200_SAMPLE = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','HINDUNILVR.NS',
    'ICICIBANK.NS','BHARTIARTL.NS','KOTAKBANK.NS','LT.NS','SBIN.NS',
    'BAJFINANCE.NS','ASIANPAINT.NS','AXISBANK.NS','MARUTI.NS','TITAN.NS',
    'NESTLEIND.NS','HCLTECH.NS','WIPRO.NS','SUNPHARMA.NS','ULTRACEMCO.NS',
    'BAJAJFINSV.NS','ONGC.NS','NTPC.NS','POWERGRID.NS','TECHM.NS',
    'TATAMOTORS.NS','INDUSINDBK.NS','DIVISLAB.NS','CIPLA.NS','JSWSTEEL.NS',
    'HINDALCO.NS','BPCL.NS','COALINDIA.NS','DRREDDY.NS','ADANIPORTS.NS',
    'BRITANNIA.NS','EICHERMOT.NS','GRASIM.NS','HEROMOTOCO.NS',
]

FEATURES = ['RSI_14','MACD_12_26_9','BBP_20_2.0','ATRr_14','EMA_20','EMA_50','volume_ratio','STOCHk_14_3_3']
TARGET = 'target'

In [ ]:
def build_features(symbol: str) -> pd.DataFrame:
    ticker = yf.Ticker(symbol)
    df = ticker.history(period='5y', interval='1d')
    if df.empty or len(df) < 60:
        return pd.DataFrame()
    
    df = df.reset_index()
    df.columns = [c.lower() for c in df.columns]
    df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
    
    df.ta.rsi(length=14, append=True)
    df.ta.macd(append=True)
    df.ta.bbands(length=20, append=True)
    df.ta.atr(length=14, append=True)
    df.ta.ema(length=20, append=True)
    df.ta.ema(length=50, append=True)
    df.ta.stoch(append=True)
    
    # Target: 7-day return classification
    df['future_return'] = df['close'].shift(-7) / df['close'] - 1
    df[TARGET] = np.where(df['future_return'] > 0.02, 1,
                  np.where(df['future_return'] < -0.02, -1, 0))
    
    # Drop rows with NaN in features or target
    df = df.dropna(subset=FEATURES + [TARGET])
    df['symbol'] = symbol
    return df

In [ ]:
print('Downloading data and computing features...')
frames = []
for sym in NIFTY_200_SAMPLE:
    try:
        df = build_features(sym)
        if not df.empty:
            frames.append(df)
            print(f'  {sym}: {len(df)} rows')
    except Exception as e:
        print(f'  {sym}: FAILED — {e}')

all_data = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(all_data)}, symbols: {len(frames)}')
print('Target distribution:\n', all_data[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# Walk-forward cross-validation (time-series aware)
X = all_data[FEATURES].values
y = all_data[TARGET].values

tscv = TimeSeriesSplit(n_splits=5)
scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42,
        tree_method='gpu_hist',  # Use GPU on Kaggle
        device='cuda',
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    score = model.score(X_val, y_val)
    scores.append(score)
    print(f'Fold {fold+1}: accuracy={score:.3f}')

print(f'\nMean accuracy: {np.mean(scores):.3f} ± {np.std(scores):.3f}')

In [ ]:
# Final model trained on all data
final_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    tree_method='gpu_hist',
    device='cuda',
)
final_model.fit(X, y)

# Save model
joblib.dump(final_model, 'xgb_model.joblib')
print('Saved: xgb_model.joblib')
print('Feature importances:')
for fname, imp in sorted(zip(FEATURES, final_model.feature_importances_), key=lambda x: -x[1]):
    print(f'  {fname}: {imp:.3f}')

### Upload to backend
Download `xgb_model.joblib` from Kaggle and place at `backend/app/models/xgb_model.joblib`